<a href="https://colab.research.google.com/github/palarunava/machine-learning-courses/blob/main/machine-learning-misc/audio_feature_extracor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Record and Transcribe

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [ ]:
# model_id = "openai/whisper-large-v3-turbo"
model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    batch_size=16,  # batch size for inference - set based on your device
    torch_dtype=torch_dtype,
    device=device,
)

In [ ]:
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation", download_mode=False)
sample = dataset[0]["audio"]
print(sample)

In [ ]:
result = pipe(sample)
print(result["text"])

In [ ]:
from IPython.display import display, Javascript
from google.colab import output
import base64

def record_voice(filename='recording.wav'):
  js = Javascript('''
    async function recordAudio() {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = 'Record';
      button.style.background = 'red';
      button.style.color = 'white';
      button.style.padding = '10px';
      document.body.appendChild(div);
      div.appendChild(button);

      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      recorder.ondataavailable = (e) => chunks.push(e.data);
      recorder.onstop = async () => {
        const blob = new Blob(chunks);
        const reader = new FileReader();
        reader.readAsDataURL(blob);
        reader.onloadend = () => {
          window.callback(reader.result);
        };
      };

      button.onclick = () => {
        if (recorder.state === 'inactive') {
          recorder.start();
          button.textContent = 'Stop Recording';
        } else {
          recorder.stop();
          button.textContent = 'Done!';
        }
      };

      return new Promise((resolve) => {
        window.callback = resolve;
      });
    }
  ''')
  display(js)
  data = output.eval_js('recordAudio()')
  binary = base64.b64decode(data.split(',')[1])

  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# Run the function
audio_file = record_voice()
print(f"Saved as {audio_file}")

In [ ]:
from IPython.display import Audio
Audio(audio_file)

In [ ]:
generate_kwargs = {
    "language": "english",
    "condition_on_prev_tokens": False,
    "compression_ratio_threshold": 1.35,  # zlib compression ratio threshold (in token space)
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.6,
    "return_timestamps": True,
}

# result = pipe(sample, return_timestamps=True)
result = pipe('recording.wav', generate_kwargs=args)
print(result["text"])

In [ ]:
#@title Part 1: Explicit Audio Chunking with WavLM Embeddings
import torch
import gc
import numpy as np
from transformers import AutoFeatureExtractor, WavLMModel
from datasets import load_dataset

# 1. Clear memory & setup hardware configuration
gc.collect()
torch.cuda.empty_cache()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
# Note: WavLM is typically stable in float32; float16 can be used on modern GPUs
torch_dtype = torch.float32

model_id = "microsoft/wavlm-base-plus"

# 2. Load WavLM Native Components
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model = WavLMModel.from_pretrained(model_id).to(device, dtype=torch_dtype)

# 3. Pull the sample long audio array
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation", download_mode=False)
sr = 16000
audio_array = dataset[0]["audio"]["array"]

# 4. Define Chunking and Stride rules (matching your original spec)
CHUNK_DURATION = 30
OVERLAP_DURATION = 5

chunk_samples = CHUNK_DURATION * sr
overlap_samples = OVERLAP_DURATION * sr
stride_samples = chunk_samples - overlap_samples

chunked_embeddings = []

print(f"Total audio length: {len(audio_array)/sr:.2f} seconds")
print("Extracting acoustic chunk embeddings manually...\n")

# 5. The Sliding Window Loop
for start_idx in range(0, len(audio_array), stride_samples):
    end_idx = start_idx + chunk_samples
    chunk = audio_array[start_idx:end_idx]

    global_start_time = start_idx / sr
    global_end_time = min(end_idx / sr, len(audio_array) / sr)

    if len(chunk) < sr * 0.5: # Skip tiny leftovers
        continue

    # Pad trailing tail chunk with zeros if it falls short of 30 seconds
    if len(chunk) < chunk_samples:
        chunk = np.pad(chunk, (0, chunk_samples - len(chunk)), 'constant')

    # Extract structural acoustic features
    inputs = feature_extractor(chunk, sampling_rate=sr, return_tensors="pt")
    input_values = inputs.input_values.to(device, dtype=torch_dtype)

    with torch.no_grad():
        outputs = model(input_values)
        # Sequence shape: [batch, sequence_length, 768]
        last_hidden_states = outputs.last_hidden_state

        # Mean Pooling: Collapse time dimension to catch holistic properties of the window
        mean_pooled = torch.mean(last_hidden_states, dim=1).squeeze()
        embedding = mean_pooled.cpu().numpy()

    chunked_embeddings.append({
        "chunk_index": len(chunked_embeddings),
        "global_window_seconds": (global_start_time, global_end_time),
        "embedding": embedding # 768-dimensional float array
    })

    print(f"Processed Chunk {chunked_embeddings[-1]['chunk_index']}: Window {global_start_time:.1f}s to {global_end_time:.1f}s | Shape: {embedding.shape}")

# To represent the entire track as a single, holistic master vector:
all_vectors = [c["embedding"] for c in chunked_embeddings]
master_audio_embedding = np.mean(all_vectors, axis=0)
print(f"\nFinal global track embedding generated with shape: {master_audio_embedding.shape}")

In [ ]:
#@title Part 2: Measuring Similarity (Qualitative vs. Quantitative)
import torch.nn.functional as F

# Helper function to extract a pooled embedding vector from raw audio
def get_holistic_embedding(audio_data, sampling_rate=16000):
    inputs = feature_extractor(audio_data, sampling_rate=sampling_rate, return_tensors="pt")
    inputs = inputs.input_values.to(device, dtype=torch_dtype)
    with torch.no_grad():
        outputs = model(inputs)
        return torch.mean(outputs.last_hidden_state, dim=1) # Keeps batch dim for F.cosine_similarity

# --- Scenario A: Qualitative Similarity Example ---
# Two audio slices from the exact same speaker, recording environment, and background noise levels,
# but they are saying entirely different sentences.
audio_speaker1_phraseA = audio_array[0 : 10 * sr]             # First 10 seconds
audio_speaker1_phraseB = audio_array[15 * sr : 25 * sr]       # A completely different 10 seconds

emb_qual_1 = get_holistic_embedding(audio_speaker1_phraseA)
emb_qual_2 = get_holistic_embedding(audio_speaker1_phraseB)

qualitative_similarity = F.cosine_similarity(emb_qual_1, emb_qual_2).item()


# --- Scenario B: Quantitative Similarity Example ---
# Two audio clips with matching cadence properties. For demonstration, we simulate
# a quantitative variation (like an exact pitch shift or clean speed alteration)
# to show how structural audio manipulation preserves high quantitative feature overlaps.
audio_base = audio_array[0 : 15 * sr]

# Simulate a clean quantitative shift (e.g., applying a minor pitch modulation natively)
# For code safety without extra external dependencies, we use a basic array operation
audio_pitched = np.ascontiguousarray(audio_base * 0.95)

emb_quant_1 = get_holistic_embedding(audio_base)
emb_quant_2 = get_holistic_embedding(audio_pitched)

quantitative_similarity = F.cosine_similarity(emb_quant_1, emb_quant_2).item()


# --- Print Metric Diagnostics ---
print("\n--- EMBEDDING SIMILARITY ANALYSIS ---")
print(f"Qualitative Match (Same Speaker / Context, Different Words): {qualitative_similarity:.4f}")
print(f"Quantitative Match (Same Pacing / Signal Structure Alteration): {quantitative_similarity:.4f}")

In [ ]:
#@title DLAI Suggestion for Interview Rating pipeline
import torch
import numpy as np
from transformers import pipeline

# =====================================================================
# STEP 1: AUDIO TRANSCRIPTION & TIME ANALYTICS (Using Whisper)
# =====================================================================

# Initialize Whisper pipeline with chunking and timestamp options enabled
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device=device
)

def analyze_interview_audio(audio_path_or_array):
    print("Processing audio with Whisper...")
    # Retrieve transcription with word-level timestamps
    result = pipe(
        audio_path_or_array,
        return_timestamps="word"
    )

    text = result["text"]
    chunks = result["chunks"]

    # Programmatically calculate delivery metrics
    total_words = len([c for c in chunks if "text" in c])

    # Calculate silences/pauses (gaps between words greater than 1.5 seconds)
    pauses = 0
    for i in range(len(chunks) - 1):
        end_current = chunks[i]["timestamp"][1]
        start_next = chunks[i+1]["timestamp"][0]
        if end_current is not None and start_next is not None:
            if (start_next - end_current) > 1.5:
                pauses += 1

    total_duration = chunks[-1]["timestamp"][1] if chunks else 1.0
    words_per_minute = (total_words / total_duration) * 60

    return {
        "transcript": text,
        "metrics": {
            "words_per_minute": round(words_per_minute, 1),
            "long_pauses_count": pauses,
            "total_duration_seconds": round(total_duration, 2)
        }
    }

# Simulating processing on a dummy sample
# sample_audio = "candidate_answer.mp3"
# audio_analysis = analyze_interview_audio(sample_audio)

# Mocked output for the sake of the structural demonstration:
audio_analysis = {
    "transcript": "A REST API is stateless... um, meaning that the server does not store any session data about the client. Every request must, uh, contain all the information needed.",
    "metrics": {
        "words_per_minute": 110.5,
        "long_pauses_count": 2,
        "total_duration_seconds": 15.4
    }
}

print("\n--- Audio Analytics Extracted ---")
print(audio_analysis)

# =====================================================================
# STEP 2: KNOWLEDGE VERIFICATION (LLM-as-a-Judge Prompt)
# =====================================================================

# This is how you would construct your prompt to a foundational LLM
# to keep it grounded, objective, and outputting structured JSON.

reference_answer_from_rag = """
A REST API must be stateless. The server should not store any context or session data about the client.
Each individual request from a client must contain all necessary information and authentication details to understand and process it.
"""

llm_judge_prompt = f"""
You are an expert technical interviewer acting as a strict, objective grading judge.
Analyze the candidate's transcript against the Reference Answer retrieved from our Knowledge Base.

[Reference Answer]
{reference_answer_from_rag}

[Candidate Transcript]
{audio_analysis['transcript']}

[Grading Instructions]
1. Evaluate if the logical sequence matches and if the fundamental points are fully covered.
2. To prevent hallucination or being overly generous, you MUST extract a direct quote from the Candidate Transcript to prove a point was met.
3. Ignore minor verbal fillers like "um" or "uh" (focus entirely on semantic correctness).

Provide your final assessment strictly in the following JSON format:
{{
  "technical_accuracy_score": <int from 0 to 100>,
  "points_covered": [
     {{"point": "Statelessness criteria", "status": "Met/Unmet", "justifying_quote": "string or null"}}
  ],
  "logical_progression_rating": "Excellent/Fair/Poor",
  "constructive_feedback": "string"
}}
"""

print("\n--- Generated LLM-as-a-Judge System Prompt ---")
print(llm_judge_prompt)

In [ ]:
#@title Extracting Fluency and Utterances metrics from audio sample
import torch
from transformers import pipeline

# 1. Setup the pipeline with word-level timestamps explicitly enabled
device = "cuda" if torch.cuda.is_available() else "cpu"
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device=device
)

# 2. Process your interview audio segment
# result = asr_pipe("candidate_interview_clip.wav", return_timestamps="word")

# Mocking a raw word-level timestamp structure returned by Whisper for demonstration:
mock_whisper_output = {
    "text": "A REST API is um, stateless... meaning that, meaning that the server does not store data.",
    "chunks": [
        {"text": "A", "timestamp": (0.1, 0.3)},
        {"text": " REST", "timestamp": (0.3, 0.6)},
        {"text": " API", "timestamp": (0.6, 1.0)},
        {"text": " is", "timestamp": (1.0, 1.2)},
        {"text": " um,", "timestamp": (1.2, 1.6)},          # Filler utterance
        {"text": " stateless...", "timestamp": (3.8, 4.5)}, # Note the huge time jump here (2.2s pause)
        {"text": " meaning", "timestamp": (4.5, 4.8)},      # Repetition start
        {"text": " that,", "timestamp": (4.8, 5.0)},
        {"text": " meaning", "timestamp": (5.0, 5.3)},      # Repetition end
        {"text": " that", "timestamp": (5.3, 5.5)},
        {"text": " the", "timestamp": (5.5, 5.7)},
        {"text": " server", "timestamp": (5.7, 6.1)},
    ]
}

def calculate_qualitative_metrics(whisper_data):
    chunks = whisper_data["chunks"]

    total_words = len(chunks)
    duration = chunks[-1]["timestamp"][1] - chunks[0]["timestamp"][0]

    # 1. Calculate Pacing (WPM)
    wpm = (total_words / duration) * 60

    # 2. Track Hesitations (Pauses > 1.5 seconds)
    pauses_count = 0
    filler_utterances = 0
    words_list = []
    repetitions = 0

    # List of targeted filler tokens to monitor
    filler_dictionary = ["um", "uh", "ah", "like"]

    for i in range(len(chunks)):
        clean_word = chunks[i]["text"].strip().lower().replace(",", "").replace(".", "")
        words_list.append(clean_word)

        # Check for verbal tics/utterances
        if clean_word in filler_dictionary:
            filler_utterances += 1

        # Check for structural silences between words
        if i < len(chunks) - 1:
            current_word_end = chunks[i]["timestamp"][1]
            next_word_start = chunks[i+1]["timestamp"][0]

            if current_word_end and next_word_start:
                if (next_word_start - current_word_end) > 1.5:
                    pauses_count += 1

        # Basic check for quick structural phrase repeats (e.g., "meaning that, meaning that")
        if i >= 2:
            if words_list[i] == words_list[i-2] and words_list[i-1] == words_list[i-3]:
                repetitions += 1

    return {
        "pace_wpm": round(wpm, 1),
        "hesitation_pauses": pauses_count,
        "filler_utterance_count": filler_utterances,
        "phrase_repetitions": repetitions,
        "speech_fluidity_rating": "Fluid" if pauses_count == 0 and filler_utterances < 2 else "Fragmented"
    }

metrics = calculate_qualitative_metrics(mock_whisper_output)

print("--- QUALITATIVE AUDIO METRICS ---")
for key, val in metrics.items():
    print(f"{key.replace('_', ' ').title()}: {val}")

In [ ]:
!pip install librosa

In [ ]:
#@title Audio classifier
import librosa
import torch
import numpy as np
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

# 1. Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Load a pre-trained model fine-tuned to recognize tone, pacing, and speech emotion
# This model uses its internal 1D-CNN filters to automatically map acoustic signatures
# model_id = "ehsanaghaei/wav2vec2-base-Speech_Emotion_Recognition"
# model_id = "harshit345/xlsr-wav2vec2-speech-emotion-recognition" # fine-tuned model to extract sentiment from a audio clip
model_id = "microsoft/wavlm-large"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model = AutoModelForAudioClassification.from_pretrained(model_id).to(device)

# 3. Simulate processing a candidate's response audio array (16kHz)
# (In production, replace this with your actual audio array segment)
sampling_rate = 16000
file_path = "audio_indian.mp3"
audio_array, orig_sr = librosa.load(file_path, sr=sampling_rate)

# Ensure data type is float32 (which is what Hugging Face models expect)
audio_array = audio_array.astype(np.float32)

# mock_audio_duration_seconds = 5
# dummy_audio = np.random.uniform(-0.1, 0.1, sampling_rate * mock_audio_duration_seconds).astype(np.float32)

# 4. Extract raw features using the model's preprocessing configuration
# inputs = feature_extractor(dummy_audio, sampling_rate=sampling_rate, return_tensors="pt")
inputs = feature_extractor(audio_array, sampling_rate=sampling_rate, return_tensors="pt")
input_values = inputs.input_values.to(device)

# 5. Pass through the pre-trained neural filters
with torch.no_grad():
    logits = model(input_values).logits

    # Apply Softmax to turn the raw outputs into percentage probabilities
    probabilities = torch.nn.functional.softmax(logits, dim=-1).squeeze().cpu().numpy()

# 6. Map probabilities to the learned vocal profiles
labels = model.config.id2label

print("--- AUTOMATIC ACOUSTIC ANALYSIS ---")
for idx, prob in enumerate(probabilities):
    label_name = labels.get(idx, f"Class {idx}")
    print(f"Confidence score for trait [{label_name}]: {prob * 100:.2f}%")

In [ ]:
#@title Audio feature extractor
import librosa
import torch
import numpy as np
from transformers import AutoFeatureExtractor, AutoModel

# 1. Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Load a pre-trained model fine-tuned to recognize tone, pacing, and speech emotion
# This model uses its internal 1D-CNN filters to automatically map acoustic signatures
# model_id = "ehsanaghaei/wav2vec2-base-Speech_Emotion_Recognition"
model_id = "microsoft/wavlm-large"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to(device)

# 3. Simulate processing a candidate's response audio array (16kHz)
# (In production, replace this with your actual audio array segment)
sampling_rate = 16000
file_path = "audio_indian.mp3"
audio_array, orig_sr = librosa.load(file_path, sr=sampling_rate)
# Ensure data type is float32 (which is what Hugging Face models expect)
audio_array = audio_array.astype(np.float32)

# 4. Extract raw features using the model's preprocessing configuration
# inputs = feature_extractor(dummy_audio, sampling_rate=sampling_rate, return_tensors="pt")
inputs = feature_extractor(audio_array, sampling_rate=sampling_rate, return_tensors="pt")
input_values = inputs.input_values.to(device)

# 5. Pass through the pre-trained neural filters
with torch.no_grad():
    outputs = model(input_values)

    # Extract the hidden states from the final layer of the transformer backbone
    # Shape: [batch_size, sequence_length, 1024]
    last_hidden_states = outputs.last_hidden_state

    # Perform mean pooling across the time (sequence) dimension to get a single vector
    embeddings = torch.mean(last_hidden_states, dim=1).squeeze().cpu().numpy()

print("--- VECTORIZATION COMPLETE ---")
print(f"Generated embedding vector with shape: {embeddings.shape}")
print(f'Embedding vector: {embeddings}')

In [ ]:
#@title Acoustic feature extraction with chunking
import librosa
import torch
import numpy as np
from transformers import AutoFeatureExtractor, AutoModel

# 1. Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Load a pre-trained model fine-tuned to recognize tone, pacing, and speech emotion
# This model uses its internal 1D-CNN filters to automatically map acoustic signatures
# model_id = "ehsanaghaei/wav2vec2-base-Speech_Emotion_Recognition"
model_id = "microsoft/wavlm-large"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to(device)

# 3. Simulate processing a candidate's response audio array (16kHz)
# (In production, replace this with your actual audio array segment)
sampling_rate = 16000
file_path = "audio_indian.mp3"
audio_array, orig_sr = librosa.load(file_path, sr=sampling_rate)
# Ensure data type is float32 (which is what Hugging Face models expect)
audio_array = audio_array.astype(np.float32)

# Define chunking boundaries (e.g., 10 seconds per chunk)
chunk_duration = 10
chunk_size = sampling_rate * chunk_duration

all_embeddings = []

print("--- STARTING BATCH PROCESSING ---")
# Loop through the 30-minute audio in 10-second intervals
for i in range(0, len(audio_array), chunk_size):
    chunk = audio_array[i : i + chunk_size]

    # 4. Extract raw features using the model's preprocessing configuration
    # Preprocess the individual chunk
    inputs = feature_extractor(chunk, sampling_rate=sampling_rate, return_tensors="pt")
    input_values = inputs.input_values.to(device)

    # 5. Pass through the pre-trained neural filters
    with torch.no_grad():
        outputs = model(input_values)
        last_hidden_states = outputs.last_hidden_state

        # Mean pool this specific chunk
        chunk_embedding = torch.mean(last_hidden_states, dim=1).squeeze().cpu().numpy()
        all_embeddings.append(chunk_embedding)

# Step 6: Average all chunks together to get one final 1024 vector for the entire 30 mins
final_embedding = np.mean(all_embeddings, axis=0)

print("--- COMPLETE ---")
print(f"Final aggregated embedding shape: {final_embedding.shape}")
print(f'Embedding vector: {final_embedding}')

# import numpy as np
# np.set_printoptions(threshold=np.inf)
# print(final_embedding)
# np.set_printoptions(threshold=1000) # Reset to default or a reasonable value if needed for subsequent outputs

In [ ]:
#@title Transcribe audio with word-level timestamps
import torch
import gc
import librosa
import numpy as np
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline

# 1. Clear memory & setup hardware configuration
gc.collect()
torch.cuda.empty_cache()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

# 2. Load the native model components
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa"
).to(device)

sampling_rate = 16000
file_path = "audio_indian.mp3"
audio_array, orig_sr = librosa.load(file_path, sr=sampling_rate)
# Ensure data type is float32 (which is what Hugging Face models expect)
audio_array = audio_array.astype(np.float32)

# 4. Define your Chunking and Stride rules
CHUNK_DURATION = 30
OVERLAP_DURATION = 5

chunk_samples = CHUNK_DURATION * sampling_rate
overlap_samples = OVERLAP_DURATION * sampling_rate
stride_samples = chunk_samples - overlap_samples

# This structure will hold your explicit chunk-wise data
chunked_results = []

print(f"Total audio length: {len(audio_array)/sampling_rate:.2f} seconds")
print("Processing explicit chunks manually...\n")

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=torch_dtype,
    device=device,
)

# 5. The Sliding Window Loop
for start_idx in range(0, len(audio_array), stride_samples):
    end_idx = start_idx + chunk_samples
    chunk = audio_array[start_idx:end_idx]

    # Calculate the global clock positioning for metadata records
    global_start_time = start_idx / sampling_rate
    global_end_time = min(end_idx / sampling_rate, len(audio_array) / sampling_rate)

    if len(chunk) < sampling_rate * 0.5: # Skip tiny leftover audio shards
        continue

    # Pad trailing tail chunk with zeros if it falls short of 30 seconds
    if len(chunk) < chunk_samples:
        chunk = np.pad(chunk, (0, chunk_samples - len(chunk)), 'constant')

    print(f'Running the pipeline on chunk of length {len(chunk)}')
    result = asr_pipe(
        chunk,
        return_timestamps="word",
        generate_kwargs={"temperature": 0.0, "language": "en"},
    )

    # FIX 1: Generate BOTH input_features and attention_mask
    # inputs = processor(chunk, sampling_rate=sampling_rate, return_attention_mask=True, return_tensors="pt")
    # input_features = inputs.input_features.to(device, dtype=torch_dtype)
    # attention_mask = inputs.attention_mask.to(device)

    # Generate text & word timestamps for THIS SPECIFIC CHUNK ONLY
    # with torch.no_grad():
    #     # FIX 2: Set return_dict_in_generate=True so we can safely unpack outputs
    #     outputs = model.generate(
    #         input_features=input_features,
    #         attention_mask=attention_mask,
    #         return_timestamps='word',
    #         return_dict_in_generate=True, # Wraps outputs in a safe dictionary structure,
    #         temperature=0.0,
    #         language="en",
    #     )

    # # FIX 3: Unpack the generated text token ids cleanly
    # predicted_ids = outputs["sequences"]
    # transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Word-level timestamps come from processor's decode with offsets
    # decoded = processor.batch_decode(
    #     predicted_ids,
    #     skip_special_tokens=True,
    #     output_offsets=True   # <-- this gives you word-level start/end offsets
    # )[0]

    # word_offsets = decoded["offsets"]

    # words_global = [
    #     {
    #         "text": w["text"].strip(),
    #         "start": round(w["timestamp"][0] + global_start_time, 2),
    #         "end": round(w["timestamp"][1] + global_start_time, 2),
    #     }
    #     for w in word_offsets
    #     if w["timestamp"][0] is not None and w["timestamp"][1] is not None
    # ]

    # Pull word-level timestamps from segments instead of output_offsets
    # words_global = []
    # segments = outputs.get("segments")

    # if segments is not None:
    #     # segments[0] = list of segment dicts for batch item 0
    #     for seg in segments[0]:
    #         # 'result' holds the per-token/word timing info depending on transformers version
    #         seg_text = processor.decode(seg["tokens"], skip_special_tokens=True).strip()
    #         seg_start = seg["start"].item() if torch.is_tensor(seg["start"]) else seg["start"]
    #         seg_end = seg["end"].item() if torch.is_tensor(seg["end"]) else seg["end"]
    #         words_global.append({
    #             "text": seg_text,
    #             "start": round(seg_start + global_start_time, 2),
    #             "end": round(seg_end + global_start_time, 2),
    #         })

    transcription = result["text"].strip()
    words_global = [
        {
            "text": w["text"].strip(),
            "start": round(w["timestamp"][0] + global_start_time, 2),
            "end": round(w["timestamp"][1] + global_start_time, 2),
        }
        for w in result["chunks"]
        if w["timestamp"][0] is not None and w["timestamp"][1] is not None
    ]

    # Append the structured result for this explicit block
    chunked_results.append({
        "chunk_index": len(chunked_results),
        "global_window_seconds": (global_start_time, global_end_time),
        "text": transcription.strip(),
        "words": words_global,   # <-- new field
    })

    print(f"Processed Chunk {chunked_results[-1]['chunk_index']}: Window {global_start_time:.1f}s to {global_end_time:.1f}s")

    # --- Memory hygiene: free GPU tensors each iteration ---
    # del inputs, input_features, attention_mask, outputs, predicted_ids
    del result, transcription, words_global
    gc.collect()
    torch.cuda.empty_cache()

# 6. Inspect your isolated chunk-wise data structure
print("\n--- VIEW OF MANUALLY SEPARATED CHUNKS ---")
import pprint
pprint.pprint(chunked_results)

# import json
# print(json.dumps(chunked_results, indent=4))

1. Pacing & Speed FeaturesThese metrics tell you how fast, rushed, or deliberate a speaker's articulation is.
  * Words Per Minute (WPM): Total words divided by the duration of the audio clip.
  * Instantaneous Speaking Rate: Calculated by taking the duration of an individual word ($\text{end} - \text{start}$) or a short phrase window. Rushed words have incredibly tight windows.
  * Articulatory Trajectory (Acceleration/Deceleration): Tracking how WPM fluctuates over the course of the 2 minutes. Does the candidate accelerate when nervous?
2. Pause & Fluency FeaturesTracking the "silence" between the words yields incredible insights into a speaker's hesitation or confidence.
  * Inter-word Silence (Pauses): Calculated by measuring the gap between a word's end timestamp and the next word's start timestamp ($\text{start}_{n+1} - \text{end}_n$).
  * Micro-pauses vs. Gross Pauses:
    - Micro-pauses (under 0.2 seconds) are natural linguistic boundaries.
    - Gross pauses (greater than 0.5 to 2.0+ seconds) indicate cognitive loading, hesitation, or freezing.Speech-to-Pause Ratio: The total amount of time spent actually pronouncing words versus the total time spent in silence.
3. Disfluency & Filler Word Features⚠️ Critical Whisper Warning: Standard OpenAI Whisper is trained to generate clean, readable text. By default, its internal decoder actively strips out filler words like "uh", "um", "ah", or repetitive stammers to optimize transcripts.  If you want to pull filler words out of Whisper, you have to use a workaround:
  * The Prompting Trick: Pass an initial_prompt="Umm, uh, like, okay." to the Whisper processor. This signals the model that it is acceptable to output disfluencies.
  * Alternative Tooling: Use CrisperWhisper or WhisperX (which handles forced phonetic alignment) to ensure filler words aren't dropped.<br>
  Once enabled, you can map:
  * Filler Word Frequency (Disfluency Count): Counting the occurrence of explicit tokens like "um", "uh", "like", "so", or "you know".
  * Filler Injection Rate: The percentage of total words that are filler items.
  * Hesitation Placement: Detecting if fillers appear at structural points (e.g., at the start of a sentence while planning a thought) or mid-sentence (indicating a struggle to find vocabulary).
4. Semantic & Text-Based FeaturesOnce you have the text string itself, you can pass it to downstream NLP tools (like a lightweight BERT model or an LLM) to get semantic features:
  * Vocabulary Diversity (Type-Token Ratio): The ratio of unique words to total words spoken. High diversity usually suggests high fluency and vocabulary command.
  * Sentiment and Tone Shift: Evaluating how the literal meaning of the words changes throughout the response.
  * Readability / Grade Level (Flesch-Kincaid): Assessing structural complexity based on word choice and sentence lengths calculated via the timestamp boundaries.

## Acoustic Features

In [ ]:
#@title Load acoustic features
import numpy as np
raw_string = """
[
 -7.42689818e-02  2.63349619e-02  1.94504447e-02  2.23735366e-02
  7.09868148e-02 -4.30903696e-02 -4.22102772e-02  3.50059196e-02
 -5.29581383e-02 -9.86463502e-02  2.45622713e-02  2.57360116e-02
  7.44227273e-03 -7.58020580e-02 -4.43631262e-02 -1.12089999e-02
 -1.33255906e-02 -9.75829214e-02  1.37916673e-02 -1.98458228e-02
 -2.09927242e-02  3.82931568e-02  2.29657954e-03  4.48580049e-02
 -5.70490025e-02 -2.07130238e-02  7.25623518e-02 -4.61763106e-02
 -8.38520601e-02  3.12636048e-02  1.63706280e-02 -6.65976433e-03
 -4.56578732e-02 -1.52820991e-02  8.15208256e-02 -1.42952520e-02
  4.39999551e-02  8.40277523e-02  2.78322976e-02 -4.50062798e-03
  6.49661720e-02  1.53156267e-02  4.52072658e-02 -2.81569827e-02
 -1.25491053e-01  3.94895598e-02  3.16821411e-02  1.13003617e-02
  7.21270144e-02 -4.24765535e-02 -9.24544875e-03  4.14552214e-03
  2.13173497e-02  2.65363120e-02  2.31057908e-02 -8.99789631e-02
 -1.25428075e-02 -2.18951143e-02 -5.06119467e-02  1.91582330e-02
  5.36748804e-02 -3.97622846e-02 -4.61745262e-02  8.62200279e-03
 -6.75707683e-02  9.73186642e-02 -1.41393244e-02 -1.35758962e-03
 -4.43414412e-02 -1.45736635e-02  5.73803969e-02  1.20351799e-02
  3.91279906e-02  6.28874227e-02 -8.01207218e-03 -1.81471265e-03
 -4.57682870e-02  6.30866885e-02 -5.93418069e-02 -1.01179071e-02
 -4.08451110e-02  6.12416863e-03  8.17687809e-02 -6.84877709e-02
  6.61198096e-03  2.58042347e-02  1.36049883e-02  3.67792323e-02
  6.87490497e-03 -4.92288284e-02 -5.22010922e-02 -3.66813391e-02
  5.95572330e-02 -4.07310687e-02 -1.11497135e-03 -6.20376617e-02
  4.20180559e-02 -3.94981876e-02 -5.64221777e-02 -2.22103857e-02
  9.38506518e-03 -3.32208760e-02  5.84837869e-02  5.98278642e-02
 -2.46937349e-02  2.78119394e-03 -4.49409038e-02  5.31004593e-02
 -4.37311381e-02  7.18172342e-02  3.09346952e-02  3.09657641e-02
 -8.28583613e-02  2.75885426e-02 -5.89488931e-02 -1.24050360e-02
 -2.57497318e-02  4.52083610e-02  1.76091082e-02  4.83330451e-02
  1.22309802e-03 -1.15895197e-01 -3.36238071e-02 -4.94704070e-03
  3.88692245e-02  1.76748652e-02  7.20653459e-02  1.03926487e-01
 -5.28865121e-02  2.46428736e-02  2.73654275e-02  3.29323821e-02
  4.95598912e-02  3.93144116e-02 -5.04183322e-02 -2.27288716e-02
  5.32351583e-02  3.78446952e-02  7.13122189e-02 -1.27523057e-02
  4.91425470e-02  4.35776301e-02  3.50676626e-02  5.52571416e-02
 -4.40626182e-02 -2.43749879e-02  1.48077849e-02  2.20262306e-03
 -3.21351513e-02 -4.50080168e-03  2.77490518e-03  7.98877031e-02
 -8.87844060e-03 -2.57967822e-02  2.60722172e-03  1.44820113e-03
  5.96556216e-02  7.73069784e-02  2.50430424e-02 -1.07790932e-01
  5.44541813e-02  6.01694919e-03 -2.28448920e-02 -6.13728166e-02
  5.47232442e-02  4.41514477e-02  3.07699163e-02 -4.41698022e-02
  4.38389517e-02 -3.34124714e-02 -4.12732735e-02 -1.30178463e-02
  1.48905264e-02  1.78680569e-02 -9.47866309e-03  3.69587019e-02
 -2.16161627e-02 -1.65095106e-02 -3.81829664e-02 -2.85506342e-03
  1.24565456e-02  1.40393823e-02 -1.21404193e-02 -1.35307517e-02
  1.03282362e-01  2.41184011e-02  3.14567722e-02  3.94394919e-02
  4.96803923e-03 -5.44296461e-04  5.09020090e-02  2.87769344e-02
 -2.20989846e-02 -2.56301351e-02 -4.10121912e-03 -3.38739390e-03
  3.25922146e-02 -3.26273311e-03  2.08989121e-02 -1.19839404e-02
  6.08650036e-02  8.24370421e-03  2.48350315e-02  7.21443724e-03
 -1.06893338e-01 -5.20780124e-03  3.61387357e-02  1.14157647e-02
  3.44011709e-02 -1.70135405e-02 -3.15577202e-02 -4.53343578e-02
  6.35652710e-03  5.68605661e-02  1.68507285e-02 -4.54535410e-02
  7.19415490e-03 -8.14566229e-05 -5.62259704e-02 -2.39261221e-02
 -5.97224571e-03  1.23549197e-02  3.20894718e-02 -3.64326462e-02
 -2.15411521e-02  2.88046547e-03 -1.20523176e-03 -2.32910085e-02
 -5.84800765e-02 -7.48055801e-02  3.80912162e-02 -1.01612099e-02
 -4.89556156e-02 -9.08269957e-02  3.71934064e-02 -1.70615055e-02
  4.10006903e-02  1.87341142e-02 -8.42588721e-04  9.53283310e-02
 -1.12859063e-01  5.44062406e-02 -7.03563169e-02  4.08909917e-02
  3.00964303e-02  6.23075943e-03 -2.22061779e-02 -7.52349757e-03
 -2.57628150e-02  1.20137343e-02 -4.32179719e-02 -3.68521102e-02
  4.40250896e-02 -1.07293557e-02 -1.22350538e-02 -7.84853846e-02
  2.61494778e-02  5.41302608e-03 -3.88451032e-02 -1.96237070e-03
 -3.05206012e-02  1.08195789e-01 -2.57843919e-02  2.31358632e-02
 -1.19658385e-03  3.39689218e-02 -7.20347464e-02  1.42702116e-02
  5.23690842e-02 -1.71936210e-02 -4.11436372e-02 -4.32294756e-02
 -9.79252756e-02  4.48742770e-02  4.04822305e-02 -1.41644021e-02
  2.23306417e-02 -5.22238873e-02 -2.80176196e-02  2.03867373e-03
 -1.21320516e-01 -4.74245586e-02 -6.83576763e-02 -5.57482503e-02
 -3.42655443e-02 -5.13817463e-03 -8.39900225e-02 -5.50231673e-02
  5.02997637e-02  6.19277991e-02 -5.90404049e-02  3.27745229e-02
 -3.20544690e-02  2.01030187e-02 -1.05950370e-01  5.19303270e-02
 -4.35196497e-02 -4.01137993e-02  4.80856448e-02  1.18450513e-02
 -3.22573353e-03  1.96165722e-02  2.70973772e-01 -2.01395322e-02
 -7.31174946e-02  1.18726138e-02 -7.80677497e-02 -3.61923836e-02
  7.14624003e-02  2.56586261e-02 -1.03282975e-02  6.63056523e-02
  1.07045025e-02  4.61431667e-02 -5.52372411e-02 -2.12817695e-02
  3.89095470e-02 -2.61786785e-02 -9.31302831e-02 -5.92639409e-02
 -2.35121064e-02  1.18114054e-02  9.01275314e-03 -2.57669645e-03
 -4.02435027e-02 -5.65915182e-02  1.69530082e-02 -4.08852175e-02
 -3.76034603e-02  6.04054555e-02 -2.56670527e-02 -2.48927604e-02
  1.10496290e-01 -3.23995352e-02 -3.56979519e-02 -2.41513811e-02
  1.23845950e-01 -5.90389408e-02 -3.33154090e-02  7.76948184e-02
  8.00763071e-02  7.92961046e-02  9.01050568e-02 -5.62412180e-02
  8.65527093e-02 -5.14892489e-02  6.32191002e-02 -3.27586457e-02
 -6.03436902e-02  2.72774231e-03 -5.37710153e-02  2.16564741e-02
 -2.37344932e-02 -4.06934060e-02 -2.68131029e-02  2.96296552e-02
 -2.92685218e-02 -2.75601242e-02 -4.09944095e-02 -1.01729155e-01
  2.20209025e-02 -2.77636237e-02  1.65744461e-02  1.69479717e-02
  7.87919685e-02 -4.35181893e-03 -1.37352000e-03 -3.47770415e-02
  8.91077053e-03  2.93155145e-02 -4.47146259e-02 -1.26364389e-02
  1.48480004e-02 -1.84039306e-02  2.48030517e-02 -1.41914189e-02
 -3.57230939e-02  1.01788342e-03 -4.80329767e-02  2.53151786e-02
  3.68108377e-02 -1.00069214e-02  3.73692550e-02 -8.89523700e-03
  7.34101795e-03 -5.83246909e-03  4.29080538e-02  2.64509320e-01
 -6.21213727e-02  1.73895676e-02 -1.36913043e-02 -2.64959764e-02
 -5.87308146e-02  2.66966806e-03  4.03921977e-02 -1.03471741e-01
  6.73785135e-02 -1.67730618e-02 -1.01503516e-02  9.83808376e-03
 -4.27423716e-02 -7.02643692e-02  6.64208159e-02 -2.64684372e-02
 -2.98883915e-02  2.62200069e-02 -2.28870008e-02 -4.17121337e-04
 -6.34773299e-02 -4.40532006e-02 -5.10822721e-02  2.24661250e-02
 -5.35936728e-02  3.01419012e-02 -3.06167565e-02 -8.94774497e-03
  9.27093439e-03  3.32313180e-02 -1.03366515e-02 -1.36231959e-01
  6.77361041e-02 -5.19458316e-02 -8.51866826e-02 -4.26815338e-02
  9.81244817e-03 -1.62420273e-02  3.65122706e-02 -7.64504895e-02
 -5.53485192e-02  5.29021658e-02 -1.49998181e-02 -3.30002159e-02
 -1.74045516e-03  4.23773564e-02 -8.03180132e-03  8.55860859e-03
 -9.44233593e-03 -1.07675627e-01 -4.14289860e-03  1.01404302e-02
  1.09129891e-01  8.86770561e-02 -3.18744928e-02 -5.54768406e-02
 -3.80262882e-02  4.12895810e-03 -2.36250367e-02  9.18300003e-02
 -1.15426220e-02  5.74099161e-02 -7.08778901e-03  1.74075691e-03
  5.33866882e-02 -1.16805919e-02  2.25780588e-02  9.30918660e-03
  1.21788383e-02  3.13991169e-03 -5.30821681e-02 -1.02400873e-02
 -5.39602861e-02 -9.46522923e-04  1.34211089e-02  7.76834488e-02
  2.72195823e-02  3.33110243e-02  6.42029718e-02  2.18962841e-02
 -9.05199125e-02  8.78582802e-03 -7.67498370e-03  2.01404141e-03
  3.87221240e-02  1.44298896e-01 -1.68653131e-02  3.50883082e-02
 -1.55858630e-02 -2.35622358e-02  5.92574999e-02 -3.86337489e-02
  4.38139439e-02 -1.53576694e-02  7.74359331e-02 -1.03021134e-02
  1.26452213e-02  3.01718153e-02 -5.30344807e-02 -2.92971916e-02
  3.51085924e-02 -3.34052518e-02 -1.30075298e-03 -1.88471731e-02
  2.23710071e-02  8.72767344e-03 -2.84043234e-02  9.19413418e-02
 -5.90169244e-02 -2.91979928e-02  3.67604420e-02  1.76667795e-02
  1.31928753e-02  1.03135549e-01  1.00836456e-02 -2.76716449e-03
  5.59231117e-02  1.05987951e-01  9.83957434e-04  3.11749410e-02
 -6.97654719e-03  3.24244015e-02  3.70846763e-02 -2.47749221e-03
  6.01444161e-03  8.18471760e-02 -4.27167937e-02 -1.84303150e-02
  6.86850101e-02  2.30005216e-02  7.55891157e-03  2.33218092e-02
 -2.69372500e-02  4.67009172e-02  1.83291454e-02 -3.59863089e-03
 -8.20859820e-02  4.69413139e-02 -6.64261058e-02 -7.85461590e-02
 -1.01791937e-02 -4.10511419e-02 -2.28351448e-02  1.98231079e-02
  2.00221632e-02 -9.58338082e-02  4.14209999e-02 -3.88401784e-02
  4.39992324e-02  1.96778364e-02  6.79756701e-02 -4.54539247e-02
  5.11902831e-02  5.34491464e-02 -4.17095274e-02 -8.08321126e-03
  5.64301386e-02  1.77493505e-02 -1.18652219e-02  4.53794077e-02
 -1.30141033e-02 -1.07259471e-02  4.59499806e-02  6.73092753e-02
  1.07482426e-01  5.58407307e-02 -5.57199353e-03 -7.83217624e-02
  4.03299145e-02 -6.25810027e-02  1.89161729e-02  2.27654562e-03
  5.66179049e-04  1.45112509e-02 -2.62446310e-02  5.73333539e-03
 -1.28866797e-02 -6.05820045e-02  4.33480106e-02 -6.08378313e-02
 -5.47586419e-02  1.47624891e-02 -8.32616463e-02 -2.03445591e-02
 -5.16250543e-02 -5.47550507e-02 -5.32061085e-02  1.13543570e-02
 -1.60020478e-02 -8.03620890e-02  2.73192804e-02  5.76420203e-02
  4.53004576e-02  4.40902673e-02 -1.17781181e-02  6.73421472e-02
  2.32704319e-02  4.62606326e-02  1.70676075e-02 -2.18848698e-02
  4.39687539e-03  4.29803766e-02 -5.34531809e-02  2.43904506e-04
  1.46139329e-02 -1.04875956e-02 -1.28788501e-02 -1.74851827e-02
 -2.04564333e-02 -1.24914413e-02 -1.65240262e-02 -9.33259726e-05
  6.03015609e-02  5.38152419e-02 -3.12704891e-02  2.49147788e-02
 -8.41031447e-02  1.22422207e-04 -5.18600680e-02 -1.54249566e-02
 -8.82674530e-02 -2.83372216e-02  6.53879810e-03 -6.77862689e-02
  3.08216847e-02  3.37812528e-02  3.41042839e-02 -3.33179757e-02
 -4.18084674e-02  5.78604750e-02 -5.24674840e-02 -5.58925271e-02
  4.47167121e-02 -6.13934360e-02  4.25849520e-02  3.02622784e-02
 -7.56427795e-02 -4.93646823e-02 -2.91892271e-02 -3.53482761e-03
 -4.91050351e-03  3.91006805e-02 -3.39816846e-02 -2.91242730e-03
  7.73221254e-03  5.49532920e-02  2.09672824e-02  4.07429188e-02
 -9.38299596e-02 -2.13623419e-02  5.79817500e-03 -2.50306763e-02
  1.02701616e-02 -2.50163693e-02  4.35576513e-02  4.24383841e-02
 -5.87697104e-02 -1.27034590e-01  5.65864332e-03 -4.58736867e-02
  5.04003465e-03 -4.66274396e-02  1.68009344e-02 -6.87378133e-03
  2.34365333e-02 -1.49693126e-02  3.96580696e-02  1.44638559e-02
  7.75222294e-03  1.27359286e-01 -7.33786374e-02 -2.17347965e-02
  1.14801824e-02  1.72693636e-02 -3.81810442e-02  1.26777599e-02
 -4.96799387e-02 -3.61089408e-03  2.19305195e-02  1.97165720e-02
  9.16419178e-02 -7.17072636e-02 -3.95124331e-02  4.69246097e-02
  7.81395473e-03  9.01620016e-02  2.83417068e-02 -2.07389873e-02
  2.12408490e-02 -4.48480919e-02 -9.64251012e-02  1.41902976e-02
  4.28799726e-03  5.83250187e-02  3.19591388e-02 -3.77558544e-02
  2.45496258e-02  2.18444634e-02  1.34254033e-02 -1.04590347e-02
  6.22092886e-03  8.86656344e-02 -8.12079906e-02  8.76533706e-03
  7.24424794e-02  3.83675769e-02 -9.20458045e-03  3.12089883e-02
 -5.33137657e-03  3.28912362e-02 -1.06588021e-01  4.16447148e-02
 -1.04953952e-01 -3.75607871e-02  2.95489654e-02 -5.80445528e-02
  3.26470695e-02 -2.25597396e-02  1.07211266e-02 -1.83974542e-02
  1.18180960e-02 -9.54468697e-02  1.22692585e-01  1.84663031e-02
  2.40012985e-02 -3.65163311e-02 -4.28832500e-05 -7.37733394e-02
  6.38573095e-02 -7.33184963e-02  1.94460265e-02 -2.09015496e-02
  2.02762764e-02  3.61356884e-02 -5.64353205e-02  7.56293023e-03
 -9.96026099e-02  1.88485044e-03  3.22543457e-02  1.78247318e-02
 -2.55150702e-02 -2.91839126e-03 -1.48640787e-02 -1.16265826e-02
  2.87901722e-02  1.04408771e-01 -5.94553500e-02 -3.59499361e-03
 -5.33942021e-02  1.53824873e-02 -3.77016291e-02 -4.84297611e-03
  4.40463386e-02 -5.52264089e-03 -5.71171939e-03 -1.26892608e-02
  6.94641992e-02 -1.63203049e-02  2.95981262e-02 -1.97630413e-02
  2.24425569e-02  4.71594464e-03 -9.87324584e-03 -1.81858856e-02
  2.36122981e-02 -8.80800281e-03  3.38942297e-02 -2.73482632e-02
  3.25139277e-02 -1.58088461e-01 -6.89739957e-02  4.49496172e-02
 -4.09143306e-02  3.54704587e-03  1.07398992e-02  5.58335930e-02
 -3.64113972e-02 -3.35582346e-02 -4.55164239e-02 -1.83521733e-02
  2.43223328e-02 -1.28410503e-01 -4.29135822e-02  3.90442833e-02
 -1.61945559e-02 -4.86346632e-02  1.37583809e-02  1.84295289e-02
  1.83465909e-02  4.35318090e-02  2.37140935e-02 -4.28250059e-02
 -1.23617649e-02  6.76040500e-02  7.62307225e-03 -1.41541176e-02
  1.08627742e-02 -1.30983433e-02  3.29087600e-02 -2.96273315e-03
 -2.36854888e-02 -6.05278164e-02 -4.92033437e-02  5.26879057e-02
  6.68082535e-02 -6.10717200e-02  2.87523661e-02  2.26147342e-02
 -4.05981131e-02 -5.68632223e-02 -7.71526759e-03 -5.31647950e-02
  2.23267712e-02  7.79949035e-03 -2.18295734e-02  2.76231039e-02
  4.14447859e-02 -3.09036095e-02  2.34764908e-02  1.40985707e-02
 -3.33397873e-02 -4.76748385e-02 -4.64338064e-02 -2.92723216e-02
 -2.52383016e-02 -4.60655689e-02 -7.52835497e-02 -2.80393995e-02
 -1.77669004e-02 -6.60031512e-02  9.06305108e-03  1.70618407e-02
  6.48198426e-02  8.53020772e-02 -9.01739970e-02 -6.06573885e-03
  4.24707569e-02  1.81527454e-02 -9.78811458e-03  1.94325484e-02
  2.18169354e-02 -9.16778948e-03  1.70097519e-02  6.60628751e-02
 -2.49876305e-02  4.34295945e-02 -7.61333108e-02 -4.68011051e-02
  5.52484155e-01  3.61520387e-02  3.18436436e-02  5.91217540e-02
 -3.14927585e-02 -4.46658172e-02 -5.37839793e-02 -2.04731897e-02
 -2.90598553e-02  8.66840687e-03  8.00809823e-03  3.33983935e-02
 -2.23557018e-02 -4.81502078e-02 -3.02444026e-02 -6.81210533e-02
  1.55411825e-01  9.00878944e-03  4.05060276e-02  9.85227618e-03
  3.66318859e-02  4.90273163e-02 -3.96979190e-02 -1.92707293e-02
 -1.37436260e-02 -5.72438426e-02 -1.02231186e-02  1.05359009e-03
  1.41690625e-02 -8.11384246e-03 -1.18998680e-02  1.61149502e-02
 -1.89792980e-02 -1.88558474e-01  2.91224904e-02 -1.74337886e-02
 -3.65426429e-02  1.12723303e-03 -7.19265491e-02  2.53734812e-02
 -1.36479158e-02  9.70591977e-02 -2.14909781e-02  6.49566650e-02
 -1.54559426e-02 -1.06372405e-02  5.91592342e-02  6.70208083e-03
  1.74824372e-02 -6.79335045e-03 -3.49791236e-02 -3.75860021e-03
  3.21544074e-02  4.66554128e-02  2.49130931e-02  8.91327276e-04
  1.08197173e-02 -2.37281527e-02 -2.34327242e-02 -5.42821884e-02
 -4.83504795e-02  5.29266633e-02 -1.21200047e-02  3.30338515e-02
  2.45577190e-02  3.05537395e-02  8.11828524e-02  1.11611933e-01
 -3.05622406e-02  2.81548370e-02 -3.01637463e-02  2.74677407e-02
 -1.96378738e-01 -4.55514528e-02 -1.76620260e-02  2.84043737e-02
 -5.60980588e-02  3.34226969e-03  6.20058505e-04  2.91336384e-02
  4.65723909e-02  2.53329705e-02 -2.36442909e-02  2.37176251e-02
 -1.97849441e-02  2.59740762e-02 -3.54467072e-02  4.97257449e-02
  7.46562704e-02 -1.40262041e-02  1.04764149e-01  4.96973731e-02
 -1.80680584e-02 -1.53343678e-02 -1.64015349e-02 -7.60566629e-03
 -7.88870007e-02 -2.53085718e-02 -4.15299758e-02 -4.68061753e-02
  7.41732912e-03 -6.63083717e-02  1.26405768e-02 -6.00850321e-02
  9.16493759e-02  6.19821735e-02  7.55161792e-02 -9.20211244e-03
 -3.87393162e-02  5.69716133e-02  2.28769649e-02  7.16917291e-02
 -9.04668868e-03 -2.92974990e-02  1.06729597e-01  7.88485184e-02
 -4.45542391e-03  8.03911942e-04 -7.35609233e-03 -4.41076793e-03
 -3.05042695e-02 -6.87673166e-02  8.15001840e-04  2.87189204e-02
  1.08198989e-02 -2.06669606e-02 -2.32646447e-02  3.10685150e-02
 -1.78774707e-02 -8.84874165e-02 -1.14771640e-02 -1.22356219e-02
  5.10341488e-02 -5.53293154e-02  4.64672223e-02 -2.73930021e-02
  3.78235579e-02  1.76212266e-02  3.29660028e-02  7.50571489e-03
 -1.24146063e-02 -6.09117113e-02  4.39826120e-03  9.77862924e-02
 -8.81486200e-03 -3.22533548e-02 -2.79386304e-02 -1.35512780e-02
 -1.48870219e-02 -6.23193718e-02 -4.54036929e-02 -5.77870421e-02
 -2.35540681e-02 -7.35123158e-02  1.06871165e-02 -1.26325712e-01
  1.07613951e-03 -3.53010371e-02  3.14253289e-03 -5.98424822e-02
 -2.64321659e-02 -3.20486375e-03 -5.15787341e-02 -7.66522586e-02
  2.79104151e-02  5.94275519e-02  4.33823243e-02 -2.63259616e-02
  2.36085448e-02  1.76554210e-02 -3.04459650e-02  1.44848209e-02
 -9.80870053e-02 -3.83995622e-02 -4.41152863e-02 -1.32399946e-02
  3.94453146e-02  4.76289913e-02  8.16803332e-03 -4.51475121e-02
 -5.71767129e-02  2.14745719e-02  2.74941493e-02  1.11326259e-02
  3.28219868e-02  1.30760875e-02  6.03488684e-02  1.77636240e-02
]
"""
clean_string = raw_string.replace('[', '').replace(']', '')
acoustic_features = np.fromstring(clean_string, sep=' ')
acoustic_features

In [ ]:
np.save('acoustic_features.npy', acoustic_features)
print('acoustic_features array saved to acoustic_features.npy')

In [ ]:
loaded_acoustic_features = np.load('acoustic_features.npy')
print('acoustic_features array loaded from acoustic_features.npy')
print(f'Shape of loaded array: {loaded_acoustic_features.shape}')
print('First 5 elements of loaded array:')
print(loaded_acoustic_features[:5])

## Semantic features

In [ ]:
#@title Load semantic features
import json

with open('transcription.json', 'r') as f:
    transcription_data = json.load(f)

print('Transcription data loaded from transcription.json:')
print(transcription_data)
print(json.dumps(transcription_data, indent=4))

## Stitch chunks together to form complete transcript

In [ ]:
transcription_data

[{'chunk_index': 0,
  'global_window_seconds': [0.0, 30.0],
  'text': 'When we talk about gradient descent, the main difference between batch, stochastic, and mini-batch gradient descent is how much training data is used to compute each parameter update. Batch gradient descent uses the entire training data set to calculate the gradient before updating the model parameters. The advantage is that the update direction is very stable and accurate because it reflects the whole data set. The downside is that it can be computationally expensive and',
  'words': [{'text': 'When', 'start': 0.0, 'end': 0.2},
   {'text': 'we', 'start': 0.2, 'end': 0.34},
   {'text': 'talk', 'start': 0.34, 'end': 0.58},
   {'text': 'about', 'start': 0.58, 'end': 0.92},
   {'text': 'gradient', 'start': 0.92, 'end': 1.42},
   {'text': 'descent,', 'start': 1.42, 'end': 2.16},
   {'text': 'the', 'start': 2.16, 'end': 2.54},
   {'text': 'main', 'start': 2.54, 'end': 2.72},
   {'text': 'difference', 'start': 2.72, 'end'

In [ ]:
transcription_data[0]['words'][:5]

[{'text': 'When', 'start': 0.0, 'end': 0.2},
 {'text': 'we', 'start': 0.2, 'end': 0.34},
 {'text': 'talk', 'start': 0.34, 'end': 0.58},
 {'text': 'about', 'start': 0.58, 'end': 0.92},
 {'text': 'gradient', 'start': 0.92, 'end': 1.42}]

In [ ]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.0 MB/s eta 0:00:00


In [ ]:
from rapidfuzz import fuzz

threshold = 90
all_words = []
overlap_idx = 1
for i, recent_chunk in enumerate(transcription_data):
    # print(recent_chunk)
    if i == 0:
        all_words.extend(recent_chunk["words"])
        continue

    last_chunk = transcription_data[i - 1]
    # print(last_chunk)
    for _ in range(20):
        last_chunk_overlap = last_chunk['words'][-overlap_idx:]
        last_chunk_overlap_text = ' '.join([element['text'] for element in last_chunk_overlap])
        recent_chunk_overlap = recent_chunk['words'][:overlap_idx]
        recent_chunk_overlap_text = ' '.join([element['text'] for element in recent_chunk_overlap])
        overlap_idx += 1
        score = fuzz.ratio(recent_chunk_overlap_text, last_chunk_overlap_text)  # returns 0-100 instead of 0-1
        if score > threshold and last_chunk_overlap_text.split()[0] == recent_chunk_overlap_text.split()[0]:
            print(f'Last = {last_chunk_overlap_text}')
            print(f'Recent = {recent_chunk_overlap_text}')
            print(score)  # ~98
            print(overlap_idx)
            print('=' * 100)
            overlap_idx = 1
            break
# all_words

Last = the whole data set. The downside is that it can be computationally expensive and
Recent = the whole dataset. The downside is that it can be computationally expensive and slow
96.34146341463415
15
Last = immediately. However, because each update is based on only one sample,
Recent = immediately. However, because each update is based on only one sample,
100.0
12
Last = 64, or 128 records to compute each.
Recent = 64, or 128 records, to compute each
97.14285714285714
8
Last = uses all data per update, stochastic gradient descent uses is one sample.
Recent = uses all data per update, stochastic gradient descent uses one sample per
94.52054794520548
13


In [ ]:
from rapidfuzz import fuzz

def find_overlap(prev_words, next_words, window_size=5, threshold=90):
    """
    Find how many words at the end of prev_words duplicate the start
    of next_words, by comparing fixed-size windows of word_size words
    at a time. Returns the number of words in next_words to SKIP
    (the overlap length), or 0 if no overlap is found.
    """
    max_search = min(len(prev_words), len(next_words))
    window = min(window_size, max_search)

    if window == 0:
        return 0

    best_n = 0

    # Slide the comparison window across possible overlap lengths,
    # longest-first, so the first hit found is the longest valid overlap.
    for n in range(max_search, 0, -1):
        # Compare a window_size-word slice ending at position n in prev_words
        # against a window_size-word slice starting at position n in next_words.
        prev_window = prev_words[max(0, len(prev_words) - n - window + 1) : len(prev_words) - n + window] \
            if n < window else prev_words[-n:][:window]
        # Simpler: just take the first `window` words of the candidate overlap region
        prev_slice = prev_words[-n:][:window]
        next_slice = next_words[:n][:window]

        if not prev_slice or not next_slice:
            continue

        prev_text = ' '.join(w['text'] for w in prev_slice)
        next_text = ' '.join(w['text'] for w in next_slice)

        if prev_text.strip('.,').lower() == next_text.strip('.,').lower():
            print(f'Prev: {prev_text}')
            print(f'Next: {next_text}')
            print('=' * 100)
            return n

        score = fuzz.ratio(prev_text, next_text)
        if score >= threshold and prev_text.split()[0] == next_text.split()[0]:
            print(f'Prev: {prev_text}')
            print(f'Next: {next_text}')
            print(f'Score: {score}')
            print('=' * 100)
            best_n = n
            return best_n  # longest-first search => first hit is longest

    return best_n

In [ ]:
all_words = []

for i, chunk in enumerate(transcription_data):
    if i == 0:
        all_words.extend(chunk["words"])
        continue

    prev_words = transcription_data[i - 1]["words"]
    curr_words = chunk["words"]

    overlap_n = find_overlap(prev_words, curr_words, window_size=8, threshold=90)

    if overlap_n > 0:
        print(f"Chunk {i-1}->{i}: overlap of {overlap_n} words "
              f"('{' '.join(w['text'] for w in curr_words[:overlap_n])}')")

    all_words.extend(curr_words[overlap_n:])

full_text = ' '.join(w['text'] for w in all_words)
print(full_text)

Prev: the whole data set. The downside is that
Next: the whole dataset. The downside is that it
Score: 95.1219512195122
Chunk 0->1: overlap of 14 words ('the whole dataset. The downside is that it can be computationally expensive and slow')
Prev: immediately. However, because each update is based on
Next: immediately. However, because each update is based on
Chunk 1->2: overlap of 11 words ('immediately. However, because each update is based on only one sample,')
Prev: 64, or 128 records to compute each.
Next: 64, or 128 records, to compute each
Score: 97.14285714285714
Chunk 2->3: overlap of 7 words ('64, or 128 records, to compute each')
Prev: uses all data per update, stochastic gradient descent
Next: uses all data per update, stochastic gradient descent
Chunk 3->4: overlap of 12 words ('uses all data per update, stochastic gradient descent uses one sample per')
When we talk about gradient descent, the main difference between batch, stochastic, and mini -batch gradient descent is ho

In [ ]:
def stitch_chunks(chunks):
    all_words = []

    for i, chunk in enumerate(chunks):
        new_words = chunk["words"]

        if i == 0:
            all_words.extend(new_words)
            continue

        prev_chunk_end = chunks[i - 1]["global_window_seconds"][1]
        this_chunk_start = chunk["global_window_seconds"][0]
        cutoff = (this_chunk_start + prev_chunk_end) / 2

        all_words = [w for w in all_words if w["start"] < cutoff]
        all_words.extend([w for w in new_words if w["start"] >= cutoff])

    full_text = " ".join(w["text"] for w in all_words)
    return all_words, full_text

stitched_words, full_transcript = stitch_chunks(transcription_data)

print(full_transcript)
print(f"\nTotal words: {len(stitched_words)}")
print(f"Duration covered: {stitched_words[0]['start']:.2f}s to {stitched_words[-1]['end']:.2f}s")

When we talk about gradient descent, the main difference between batch, stochastic, and mini -batch gradient descent is how much training data is used to compute each parameter update. Batch gradient descent uses the entire training data set to calculate the gradient before updating the model parameters. The advantage is that the update direction is very stable and accurate because it reflects the whole data set. The downside is that it can be computationally expensive and slow for large datasets since every update requires processing all training examples. Stochastic gradient descent, or SGD, sits at the other extreme. It updates the parameters after processing just a single training example. This makes updates very fast and allows the algorithm to start learning immediately. However, because each update is based on only one sample, the gradient estimates are noisy, so the optimization path tends to fluctuate and can be less stable. Mini -batch gradient descent is a compromise between

## Extract semantic features

In [ ]:
# Hypothethical whisper output list
word_timestamps = [
    {"word": "Let's", "start": 0.5, "end": 0.8},
    {"word": "see", "start": 0.9, "end": 1.4},
    {"word": "um", "start": 2.2, "end": 2.6}, # Filler
    {"word": "next", "start": 2.8, "end": 3.1}
]

fillers = ["um", "uh", "ah", "like"]
total_words = len(word_timestamps)

# 1. Calculate Pacing
duration = word_timestamps[-1]["end"] - word_timestamps[0]["start"]
wpm = (total_words / duration) * 60

# 2. Calculate Pauses and Find Fillers
pause_durations = []
filler_count = 0

for i in range(len(word_timestamps) - 1):
    current_word = word_timestamps[i]
    next_word = word_timestamps[i+1]

    # Pause is the gap between current end and next start
    gap = next_word["start"] - current_word["end"]
    if gap > 0:
        pause_durations.append(gap)

    if current_word["word"].lower().strip(".,!?") in fillers:
        filler_count += 1

print(f"Pacing: {wpm:.1f} WPM")
print(f"Total Fillers: {filler_count}")
print(f"Average Pause Duration: {np.mean(pause_durations):.2f}s")